In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("train.txt",sep=";",header=None,names=["Text","Emotions"])

In [3]:
df.head()

,Text,Emotions
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
df.isnull().sum()

Text        0
Emotions    0
dtype: int64

In [5]:
df=df.drop_duplicates()

In [6]:
unique_emotion=df["Emotions"].unique()

In [7]:
unique_emotion

<StringArray>
['sadness', 'anger', 'love', 'surprise', 'fear', 'joy']
Length: 6, dtype: str

In [8]:
# Mapping Target Value into Numbers
unique={}
i=0
for emo in unique_emotion:
        unique[emo]=i
        i+=1
df["Emotions"]=df["Emotions"].map(unique).astype(int)
df

,Text,Emotions
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


# Text Processing

<h3>Convert to Lowercase</h3>

In [9]:
def toLowerCase(text:str):
        return text.lower()
df["Text"]=df["Text"].apply(toLowerCase)

# Remove URLs

In [10]:
import re

def remove_urls(text: str) -> str:
    pattern = r'https?://\S+|www\.\S+'
    return re.sub(pattern, '', text)

df["Text"]=df["Text"].apply(remove_urls)

# Remove Digits

In [11]:
def remove_digits(text: str):
    return re.sub(r'\d+', '', text)

df["Text"]=df["Text"].apply(remove_digits)

# Remove Emojies

In [12]:

def remove_emojis(text: str) -> str:
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F" 
        "\U0001F300-\U0001F5FF" 
        "\U0001F680-\U0001F6FF" 
        "\U0001F1E0-\U0001F1FF" 
        "\U00002700-\U000027BF" 
        "\U0001F900-\U0001F9FF" 
        "\U00002600-\U000026FF"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub('', text)

df["Text"]=df["Text"].apply(remove_emojis)

# Remove Punctuations

In [13]:
import string

def remove_punctuation(text: str) -> str:
    return text.translate(str.maketrans('', '', string.punctuation))

df["Text"]=df["Text"].apply(remove_punctuation)

In [ ]:
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))

In [15]:
def remove_stop_word(text:str)->str:
        tokens_word=word_tokenize(text)
        filtered_word=[
                word for word in tokens_word
                if word.lower() not in stop_words
        ]
        return " ".join(filtered_word)

remove_stop_word('i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake')

'go feeling hopeless damned hopeful around someone cares awake'

<h3>Remove Stop words because they don't effect in NLP</h3>

In [16]:
df["cleaned_text"]=df["Text"].apply(remove_stop_word)
df.tail()

,Text,Emotions,cleaned_text
15995,i just had a very brief time in the beanbag an...,0,brief time beanbag said anna feel like beaten
15996,i am now turning and i feel pathetic that i am...,0,turning feel pathetic still waiting tables sub...
15997,i feel strong and good overall,5,feel strong good overall
15998,i feel like this was such a rude comment and i...,1,feel like rude comment im glad
15999,i know a lot but i feel so stupid because i ca...,0,know lot feel stupid portray


<h3>Using Bag of words instead of One-hot encoding to convert documents into vectors</h3>

In [ ]:
# Ignore this code as i was testing how it works TESTING
from sklearn.feature_extraction.text import CountVectorizer
vect=CountVectorizer(ngram_range=(2,2),stop_words="english")
documents = [
    "I like Python",
    "AI is useful",
    "Machine learning is interesting"
]
model=vect.fit_transform(documents)

In [18]:
print(vect.get_feature_names_out(documents))
print(model.toarray()) # type: ignore

['ai is' 'is interesting' 'is useful' 'learning is' 'like python'
 'machine learning']
[[0 0 0 0 1 0]
 [1 0 1 0 0 0]
 [0 1 0 1 0 1]]


In [24]:
from sklearn.metrics import recall_score,precision_score,accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
BOW=CountVectorizer(ngram_range=(1,2))
TF_IDF=TfidfVectorizer(ngram_range=(1,2))
X_train, X_test, y_train, y_test = train_test_split(df["cleaned_text"], df["Emotions"], test_size=0.20, random_state=42)

<h3>Using Bag of words</h3>

In [25]:
X_train_BOW=BOW.fit_transform(X_train)
X_test_BOW=BOW.transform(X_test)

models={
        "Multinomial NaiveByes":MultinomialNB(),
        "Logistic Regression":LogisticRegression(max_iter=1000),
        "Support Vector Machine":SVC(kernel="sigmoid")
}

results=[]
for name,model in models.items():
        model.fit(X_train_BOW,y_train)
        y_predict=model.predict(X_test_BOW)
        results.append({
                "Model Name: ":name,
                "Accuracy: ":accuracy_score(y_test,y_predict),
                "Precision: ":precision_score(y_test,y_predict,average="weighted"),
                "Recall: ":recall_score(y_test,y_predict,average="weighted")
        })

In [26]:
pd.DataFrame(data=results)

,Model Name:,Accuracy:,Precision:,Recall:
0,Multinomial NaiveByes,0.735625,0.786328,0.735625
1,Logistic Regression,0.903125,0.903331,0.903125
2,Support Vector Machine,0.894375,0.895434,0.894375


<h3>Using TF-IDF</h3>

In [29]:
X_train_TF_IDF=TF_IDF.fit_transform(X_train)
X_test_TF_IDF=TF_IDF.transform(X_test)

models={
        "Multinomial NaiveByes":MultinomialNB(),
        "Logistic Regression":LogisticRegression(max_iter=1000),
        "Support Vector Machine":SVC(kernel="sigmoid")
}

results=[]
for name,model in models.items():
        model.fit(X_train_TF_IDF,y_train)
        y_predict=model.predict(X_test_TF_IDF)
        results.append({
                "Model Name":name,
                "Accuracy":accuracy_score(y_test,y_predict),
                "Precision":precision_score(y_test,y_predict,average="weighted"),
                "Recall":recall_score(y_test,y_predict,average="weighted")
        })

In [30]:
pd.DataFrame(results)

,Model Name,Accuracy,Precision,Recall
0,Multinomial NaiveByes,0.643437,0.756818,0.643437
1,Logistic Regression,0.846562,0.857806,0.846562
2,Support Vector Machine,0.898438,0.898503,0.898438


<h2>Hyperparameter Tuning to Improve the Model</h3>

In [33]:
from sklearn.model_selection import RandomizedSearchCV


models = {
    "Multinomial Naive Bayes": {
        "model": MultinomialNB(),
        "params": {
            "alpha": [0.01, 0.1, 0.5, 1, 5]
        }
    },

    "Logistic Regression": {
        "model": LogisticRegression(max_iter=1000),
        "params": {
            "C": [0.01, 0.1, 1, 10, 100],
            "solver": ["liblinear", "lbfgs"]
        }
    },

    "Support Vector Machine": {
        "model": SVC(),
        "params": {
            "C": [0.1, 1, 10],
            "kernel": ["linear", "rbf", "sigmoid"],
            "gamma": ["scale", "auto"]
        }
    }
}

results = []

for name, config in models.items():

    random_search = RandomizedSearchCV(
        estimator=config["model"],
        param_distributions=config["params"],
        n_iter=5,
        cv=5,
        scoring="accuracy",
        random_state=42,
        n_jobs=-1
    )

    random_search.fit(X_train_BOW, y_train)

    best_model = random_search.best_estimator_

    y_predict = best_model.predict(X_test_BOW)

    results.append({
        "Model Name": name,
        "Best Parameters": random_search.best_params_,
        "Accuracy": accuracy_score(y_test, y_predict),
        "Precision": precision_score(y_test, y_predict, average="weighted"),
        "Recall": recall_score(y_test, y_predict, average="weighted")
    })

pd.DataFrame(results)

c:\Users\E L I T E\miniconda3\envs\ML\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
10 fits failed out of a total of 25.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\E L I T E\miniconda3\envs\ML\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\E L I T E\miniconda3\envs\ML\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\E L I T E\miniconda3\envs\ML\Lib\site-packages\sklearn\linear_m

,Model Name,Best Parameters,Accuracy,Precision,Recall
0,Multinomial Naive Bayes,{'alpha': 0.5},0.776563,0.807894,0.776563
1,Logistic Regression,"{'solver': 'lbfgs', 'C': 10}",0.905312,0.904973,0.905312
2,Support Vector Machine,"{'kernel': 'linear', 'gamma': 'scale', 'C': 0.1}",0.903438,0.903385,0.903438


In [ ]:
from sklearn.pipeline import Pipeline
import joblib
pipeline = Pipeline([
    ("vectorizer", CountVectorizer(ngram_range=(1,2))),
    ("model", LogisticRegression(solver="lbfgs",C=10))
])
X_train_BOW=pipeline.fit_transform(X_train)
X_test_BOW=pipeline.fit(X_test)
pipeline.fit(X_train_BOW, y_train)
y_predict = pipeline.predict(X_test_BOW)
joblib.dump(pipeline, "sentiment_model.pkl")